<a href="https://colab.research.google.com/github/trevorlillywhite/HW8-NCSU-ST-554-Trevor-Lillywhite/blob/main/HW8_ST554_Lillywhite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Homework 8 - NCSU ST 554
### Author: Trevor Lillywhite
### Due Date: April 7, 2026

# Continuation from Homework 7

This assignment continues from where we left off at the end of Homework 7 so we can re-use the data ingestion, pre-processing, standardization, EDA, feature engineering, etc.

The following section, until "New Homework 8 Content", will be prior work.  Long-running code from Homework 7 will be commented out to reduce the amount of time to run the notebook again for Homework 8.

**Purpose:** Practice fitting multiple linear regression (MLR) and logistic regression models, including penalization/regulatrization.


**Data:**
+ This assignment will use the wine quality dataset from the UCI Machine Learning Repository. We will combine the `winequality-red.csv` and `winequality-white.csv` files and indicate which color each row corresponds to.
+ Although the typical response variable is `quality`, we will predict `alcohol` content for the multiple linear regression models and the type of wine for logistic regression models.
+ There are nearly a dozen input variables based on physiochemical tests (e.g., `fixed acidity`, `chlorides`, `pH`, etc.).

#### Read in and Combine Data

We will read in the two csv files separately, add a new feature (`type` as `white` or `red`), and then merge the two datasets.

In [1]:
# Import relevant modules
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_validate, \
        cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, \
        log_loss, accuracy_score
from sklearn.linear_model import LinearRegression, LassoCV, RidgeCV,\
        LogisticRegression, LogisticRegressionCV, ElasticNetCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

In [2]:
# Import red dataset
df_red = pd.read_csv('winequality-red.csv', sep=';')

# Make new feature
df_red['type'] = 1          # We will encode our colors as red = 1, white = 0

# Check first few rows
df_red.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1


It looks like the data was properly read in, and the color column was successfully added.

In [3]:
# Check data types
df_red.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
 12  type                  1599 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 162.5 KB


There are no missing values (as expected), but we should encode the `color` feature using dummy variables. We will do this after reading in the white wine dataset and combining the datasets.

In [4]:
# Import white dataset
df_white = pd.read_csv('winequality-white.csv', sep=';')

# Make new feature
df_white['type'] = 0

# Check first few rows
df_white.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6,0
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6,0
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6,0
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,0
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6,0


It looks like the data was properly read in, and the color was successfully added.

In [5]:
# Check data types
df_white.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
 12  type                  4898 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 497.6 KB


The columns and data types match, which is good. However, there are many more white wine observations, so we will need to account for this imbalanced dataset later.

In [6]:
# Combine dataframes
df = pd.concat([df_red, df_white], axis=0)

# Check around the expected transition point
df.iloc[1597:1602, :]

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5,1
1598,6.0,0.310,0.47,3.6,0.067,18.0,42.0,0.99549,3.39,0.66,11.0,6,1
0,7.0,0.270,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6,0
1,6.3,0.300,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6,0
2,8.1,0.280,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6,0


We need to reset the index!

In [7]:
# Reset index
df.reset_index(drop = True, inplace = True)

# Check again
df.iloc[1597:1602, :]

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
1597,5.9,0.645,0.12,2.0,0.075,32.0,44.0,0.99547,3.57,0.71,10.2,5,1
1598,6.0,0.310,0.47,3.6,0.067,18.0,42.0,0.99549,3.39,0.66,11.0,6,1
1599,7.0,0.270,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6,0
1600,6.3,0.300,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6,0
1601,8.1,0.280,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6,0


The index was successfully reset! The two datasets are now in one unified dataframe.

#### Split the Data

We will use stratified sampling to ensure the colors are proportionally represented in the training and test sets. We will also shuffle to remove any dataset ordinality.

At first, we will only split our dataset for regression. Classification will be split later

In [8]:
# Split training and test set
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('alcohol', axis=1),
    df['alcohol'],
    test_size = 0.20,
    shuffle = True,
    random_state = 42,
    stratify = df['type']
)

# Print observation counts and type proportions to check the split
print(len(X_train))
print(len(X_test))
print(X_train.type.value_counts())

5197
1300
type
0    3918
1    1279
Name: count, dtype: int64


The split occurred successfully. The training set has four times the number of observations as the test set (80/20 split).

The stratification also looks like it worked. The `type` proportions look correct.

## Regression Task

### Train Models

#### Multiple Linear Regression Models

Below, we will fit four different MLR models meeting the following requirements.
+ At least one will include interaction terms.
+ At least one will include some polynomial terms.
+ Cross Validation (CV) will be used to select the best MLR model.

Before deciding which models to create, let's do some quick EDA! The `seaborn pairplot()` function is a powerful tool to quickly visualize univariate and bivariate feature distributions.  


In [9]:
''' # Commented out for Homework 8 to reduce runtime
# Create pairplot
sns.pairplot(df,
             hue='type',                # color points by type (white or red)
             plot_kws={'alpha': 0.3,    # increase transparency to see points stacking up
                       's': 5}          # decrease point size
             )
'''

" # Commented out for Homework 8 to reduce runtime\n# Create pairplot\nsns.pairplot(df,\n             hue='type',                # color points by type (white or red)\n             plot_kws={'alpha': 0.3,    # increase transparency to see points stacking up\n                       's': 5}          # decrease point size\n             )\n"

_Tip: Because there are so many plots packed in, zooming in is highly recommendeded to see the axes and data. The legend at the far right-hand side shows that the blue dots correspond to white wine and the orange dots correspond to red wine._


What do we see in this wealth of data?


+ Univariate distributions (histograms along the diagonal)
    - Red wines tend to have a different unvariate distribution than white wines for most features. The features that are the least dependent on `type` include `alcohol` and `quality`.
    - This indicates that 'type' is likely an important fact to consider, though it cannot be used alone to predict `alcohol`.
+ Positive bivariate correlations
    - `density` vs. `fixed acidity` has a moderate positive correlation
    - For white wines, `density` and `residual sugar` also have a moderate positive correlation
    - Unfortunately, due to the lack of very strong bivariate correlations, we are unable to discount any features out-right for providing redundant value in relation to another feature.
+ Bivariate correlations with the target variable (`alcohol`)
    - There are no great correlations to take advantage of. The closest might be `alcohol` vs. `density`, particularly for the white wines.

Now, we will explore the correlation matrix shown as a heatmap. Highly correlated variables don't necessarily need to be used simultaneously as inputs because they communicate much of the same information.

On the other hand, any variable highly correlated with the target variable will be of great interest.

In [10]:
''' # Commented out for Homework 8 to reduce runtime
# Produce correlation heatmap
sns.heatmap(df.corr(), annot=True, annot_kws={"size": 7})
plt.title('All Wines - Correlation Matrix Heatmap')
'''

' # Commented out for Homework 8 to reduce runtime\n# Produce correlation heatmap\nsns.heatmap(df.corr(), annot=True, annot_kws={"size": 7})\nplt.title(\'All Wines - Correlation Matrix Heatmap\')\n'

What do we see here?

+ To confirm our pairplot() findings, we see that no two separate variables are highly correlated with each other. The most extreme values are:
    - -0.7 for `total sulfur dioxide` vs. `type`
    - -0.69 for `alcohol` vs. `density`
    - 0.65 for `volatile acidity` vs. `type`

+ The features most correlated with `alcohol` are:
    - density (-0.69)
    - quality (0.44)
    - residual sugar (-0.36)

These don't seem great for predictive power, so we will likely need to rely on interactions or polynomials, if chosen smartly.

To see how a third variable of `type` affects things, let's repeat this correlation matrix heatmap while splitting the data by type.

In [11]:
''' # Commented out for Homework 8 to reduce runtime
# Create correlation matrix heatmap split by wine type
#   First: red wine

sns.heatmap(df.loc[df['type'] == 1].corr(), annot=True, annot_kws={"size": 7})
plt.title('Red Wines - Correlation Matrix Heatmap')
'''

' # Commented out for Homework 8 to reduce runtime\n# Create correlation matrix heatmap split by wine type\n#   First: red wine\n\nsns.heatmap(df.loc[df[\'type\'] == 1].corr(), annot=True, annot_kws={"size": 7})\nplt.title(\'Red Wines - Correlation Matrix Heatmap\')\n'

This was useful. We see some new correlation trends sticking out.
+ Because we won't filter out features that are redundant for only some of the wine types, we won't further analyze the most extreme correlation values that are generally present in the matrix. We will focus on correlations with `alcohol` content.
+ `Alcohol` now has less extreme correlations. The strongest are with:
    - `density` (-0.5) (less significant correlation than with the full dataset)
    - `quality` (0.48) (slightly higher correlation than with the full dataset)

Now we will repeat for white wine.

In [12]:
''' # Commented out for Homework 8 to reduce runtime
# Create correlation matrix heatmap split by wine type
#   Second: white wine

sns.heatmap(df.loc[df['type'] == 0].corr(), annot=True, annot_kws={"size": 7})
plt.title('White Wines - Correlation Matrix Heatmap')
'''

' # Commented out for Homework 8 to reduce runtime\n# Create correlation matrix heatmap split by wine type\n#   Second: white wine\n\nsns.heatmap(df.loc[df[\'type\'] == 0].corr(), annot=True, annot_kws={"size": 7})\nplt.title(\'White Wines - Correlation Matrix Heatmap\')\n'

As expected, the correlation between `alcohol` and `density` had to become more significant here since it became less significant with red wines (vs. the whole dataset). Correlations with `alcohol` vs. `total sulfur dioxide` and `residual sugar` are both more pronounced now, each at -0.45.

Overall, we need to account for wine type and density, at a very minimum.

Now we have enough data to make semi-informed decisions about how to craft our models. Because these will not be regularized, we will be judicious about the number of features selected.

+ MLR Model 1 (most simple)
    - Single Features: `type`, `density`, `quality`
    - Interactions: None
    - Polynomials: None

+ MLR Model 2 (still simple but more features)
    - Single Features: `type`, `density`, `quality`, `residual sugar`, `total sulfur dioxide`
    - Interactions: None
    - Polynomials: None

+ MLR Model 3 (adding complexity with interaction terms)
    - Single Features: `type`, `density`, `quality`, `residual sugar`
    - Interactions:
        - `type` * `density`
        - `type` * `residual sugar`
        - `type` * `total sulfur dioxide`
    - Polynomials: None

+ MLR Model 4 (most complex - includes interactions and polynomials)
    - Single Features: `type`, `density`, `quality`, `residual sugar`
    - Interactions:
        - `type` * `density`
        - `type` * `residual sugar`
        - `type` * `total sulfur dioxide`
    - Polynomials:
        - `residual sugar`^2
        - `total sulfur dioxide`^2
        - `density`^2

The interaction terms were chosen based on the features whose correlation with `alcohol` was most affected by `type`.

The polynomial terms were chosen based on the visible data curvature in the `pairplot()` bivariate distributions with `alcohol`. The exponent should make the correlation more pronounced for at least one of the wine types. Because these features are all strictly positive, we don't need to worry about losing the sign of the values.

Before training the model, we will define new features for our interaction and polynomial terms and then standardize our data. Because we have not learned how to use `Pipelines` yet, there will be some data leakage within the cross-validation set. However, this is deemed acceptable for this assignment as long as we prevent data leakage from the test set.

In [13]:
# Create polynomial and interaction columns and add to dataframe

# Define columns where second-degree polynomials are desired
col_poly = ['residual sugar', 'total sulfur dioxide', 'density']

# Define columns involved in interactions
col_interact = ['type', 'density', 'residual sugar', 'total sulfur dioxide']

# Create polynomial object
poly = PolynomialFeatures(degree = 2,
                          interaction_only = False,
                          include_bias = False, )

# Create polynomial dataframe
poly_data = poly.fit_transform(df[col_poly])
poly_names = poly.get_feature_names_out(col_poly)
df_poly = pd.DataFrame(poly_data, columns=poly_names)

# Filter by actual polynomial columns (no interactions)
df_poly = df_poly[[col for col in df_poly.columns if '^2' in col]]

# Create interaction object
interact = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)

# Create interaction dataframe
interact_data = interact.fit_transform(df[col_interact])
interact_names = interact.get_feature_names_out(col_interact)
df_interact = pd.DataFrame(interact_data, columns=interact_names)

# Filter by desired interaction columns
df_interact = df_interact[[col for col in df_interact.columns if
                           'type' in col]]

# Reset indices
df_poly = df_poly.reset_index()
df_poly = df_poly.drop(columns=['index'])
df_interact = df_interact.reset_index()
df_interact = df_interact.drop(columns=['index', 'type'])

# Check new polynomial features
df_poly.head()





,residual sugar^2,total sulfur dioxide^2,density^2
0,3.61,1156.0,0.995605
1,6.76,4489.0,0.993610
2,5.29,2916.0,0.994009
3,3.61,3600.0,0.996004
4,3.61,1156.0,0.995605


In [14]:
# Check new interaction features
df_interact.head()

,type density,type residual sugar,type total sulfur dioxide
0,0.9978,1.9,34.0
1,0.9968,2.6,67.0
2,0.9970,2.3,54.0
3,0.9980,1.9,60.0
4,0.9978,1.9,34.0


It looks like we made the new interactions and polynomials that we wanted!

In [15]:
print(len(df), len(df_interact), len(df_poly))

6497 6497 6497


In [16]:
# Add new features to main DataFrame
df = pd.concat([df, df_interact, df_poly], axis=1)
df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type density,type residual sugar,type total sulfur dioxide,residual sugar^2,total sulfur dioxide^2,density^2
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1,0.9978,1.9,34.0,3.61,1156.0,0.995605
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1,0.9968,2.6,67.0,6.76,4489.0,0.993610
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,1,0.9970,2.3,54.0,5.29,2916.0,0.994009
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,1,0.9980,1.9,60.0,3.61,3600.0,0.996004
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,1,0.9978,1.9,34.0,3.61,1156.0,0.995605


In [17]:
# Check statistics for reasonability
df.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,type density,type residual sugar,type total sulfur dioxide,residual sugar^2,total sulfur dioxide^2,density^2
count,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000
mean,7.215307,0.339666,0.318633,5.443235,0.056034,30.525319,115.744574,0.994697,3.218501,0.531268,10.491801,5.818378,0.246114,0.245313,0.624835,11.436355,52.262023,16591.034824,0.989430
std,1.296434,0.164636,0.145318,4.757804,0.035034,17.749400,56.521855,0.002999,0.160787,0.148806,1.192712,0.873255,0.430779,0.429378,1.298121,25.824178,96.455471,13700.653268,0.005970
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.360000,36.000000,0.974386
25%,6.400000,0.230000,0.250000,1.800000,0.038000,17.000000,77.000000,0.992340,3.110000,0.430000,9.500000,5.000000,0.000000,0.000000,0.000000,0.000000,3.240000,5929.000000,0.984739
50%,7.000000,0.290000,0.310000,3.000000,0.047000,29.000000,118.000000,0.994890,3.210000,0.510000,10.300000,6.000000,0.000000,0.000000,0.000000,0.000000,9.000000,13924.000000,0.989806
75%,7.700000,0.400000,0.390000,8.100000,0.065000,41.000000,156.000000,0.996990,3.320000,0.600000,11.300000,6.000000,0.000000,0.000000,0.000000,0.000000,65.610000,24336.000000,0.993989
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000,9.000000,1.000000,1.003690,15.500000,289.000000,4329.640000,193600.000000,1.079479


Now that we have all of our interactions and polynomials, we will re-perform the training/test split. This will be easier than repeating all of the interaction and polynomial-generating steps separately for the training set and test set.

In [18]:
# Split training and test set (reperform after adding new variables)
X_train, X_test, y_train, y_test = train_test_split(
    df.drop('alcohol', axis=1),
    df['alcohol'],
    test_size = 0.20,
    shuffle = True,
    random_state = 42,
    stratify = df['type']
)

# Print observation counts and type proportions to check the split
print(len(X_train))
print(len(X_test))
print(X_train.type.value_counts())

5197
1300
type
0    3918
1    1279
Name: count, dtype: int64


The split was re-completed successfully.

Now, we will standardize the data using the training set.

In [19]:
# Standardize data using training set
scaler = StandardScaler()
X_train_std = pd.DataFrame(scaler.fit_transform(X_train),
                           columns=X_train.columns)
X_train_std.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type,type density,type residual sugar,type total sulfur dioxide,residual sugar^2,total sulfur dioxide^2,density^2
count,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03
mean,6.836086e-18,8.750190e-17,-6.887356e-17,-2.802795e-17,-1.777382e-17,3.999110e-17,-1.025413e-16,-4.686752e-14,-2.156101e-15,2.050826e-17,-1.517611e-16,-5.058703e-17,-1.606480e-17,-2.973697e-17,1.435578e-17,3.007878e-17,-1.011741e-16,-1.001657e-14
std,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00
min,-2.623567e+00,-1.563580e+00,-2.187852e+00,-1.010316e+00,-1.358711e+00,-1.664326e+00,-1.947703e+00,-2.522350e+00,-2.778159e+00,-2.118126e+00,-3.243464e+00,-5.713511e-01,-5.713497e-01,-4.861946e-01,-4.436919e-01,-5.191170e-01,-1.209256e+00,-2.511903e+00
25%,-6.232249e-01,-6.602944e-01,-4.723414e-01,-7.588849e-01,-5.185759e-01,-7.603909e-01,-6.682412e-01,-7.997487e-01,-6.751579e-01,-6.902768e-01,-9.466547e-01,-5.713511e-01,-5.713497e-01,-4.861946e-01,-4.436919e-01,-4.902636e-01,-7.658773e-01,-7.995423e-01
50%,-1.616075e-01,-2.989804e-01,-6.061888e-02,-5.074540e-01,-2.578442e-01,-8.243932e-02,4.257076e-02,3.989420e-02,-5.662815e-02,-1.463343e-01,2.017497e-01,-5.713511e-01,-5.713497e-01,-4.861946e-01,-4.436919e-01,-4.325569e-01,-1.911266e-01,3.835196e-02
75%,3.769461e-01,3.634287e-01,5.569648e-01,5.611276e-01,2.636192e-01,5.955123e-01,7.000718e-01,7.729158e-01,6.237546e-01,4.656009e-01,2.017497e-01,-5.713511e-01,-5.713497e-01,-4.861946e-01,-4.436919e-01,1.345922e-01,5.493780e-01,7.715845e-01
max,6.685717e+00,7.469272e+00,9.203137e+00,1.265077e+01,1.608134e+01,1.460651e+01,5.764607e+00,1.476030e+01,4.891610e+00,9.848608e+00,3.646963e+00,1.750237e+00,1.766503e+00,1.160898e+01,1.029978e+01,4.285392e+01,1.298093e+01,1.507334e+01


The standardization worked. The means for each column are extremely close to zero and the standard deviations are very near 1.0.

Now we will use the same training set statistics to scale the test set.

In [20]:
X_test_std = pd.DataFrame(scaler.transform(X_test),
                          columns=X_test.columns)
X_test_std.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,quality,type,type density,type residual sugar,type total sulfur dioxide,residual sugar^2,total sulfur dioxide^2,density^2
count,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000,1300.000000
mean,0.020199,0.000517,-0.006884,0.022324,0.019334,0.018667,0.012450,0.027255,-0.020230,-0.008623,-0.034115,0.000117,0.000236,0.022745,-0.008632,0.004323,0.021959,0.027215
std,0.986902,0.956347,0.985784,0.984109,1.072445,1.013636,1.021804,0.995357,0.972102,1.057535,1.013714,1.000454,1.000662,1.094066,0.989881,0.817937,1.021602,0.994589
min,-2.315822,-1.443141,-2.187852,-1.010316,-1.155920,-1.664326,-1.947703,-2.245800,-3.087424,-2.050133,-3.243464,-0.571351,-0.571350,-0.486195,-0.443692,-0.519117,-1.209256,-2.237601
25%,-0.546289,-0.660294,-0.472341,-0.737932,-0.518576,-0.760391,-0.739322,-0.730611,-0.675158,-0.690277,-0.946655,-0.571351,-0.571350,-0.486195,-0.443692,-0.486557,-0.810450,-0.730629
50%,-0.161608,-0.238761,-0.060619,-0.465549,-0.228874,-0.082439,0.042571,0.109864,-0.056628,-0.214327,0.201750,-0.571351,-0.571350,-0.486195,-0.443692,-0.420134,-0.191127,0.108272
75%,0.376946,0.363429,0.488344,0.582080,0.263619,0.652008,0.740055,0.747093,0.623755,0.465601,0.201750,-0.571351,-0.571350,-0.486195,-0.443692,0.150922,0.600897,0.745727
max,6.454908,4.669088,6.252459,5.484984,16.052372,6.555837,3.410042,5.204365,3.716404,9.984593,3.646963,1.750237,1.766503,11.687524,10.724885,9.481382,5.720026,5.238804


As we can see, the training set statistics were not a complete match for the test set since the standardized mean values are a little farther from zero and the standard deviations are not exactly 1.0.  But it's close, as expected. It would be suspicious if the values were too close to perfectly standardized.

Now we can subset the columns for each MLR method.

In [21]:
# Subset training and test sets for each MLR
cols_mlr1 = ['type','density','quality']
X_train_mlr1 = X_train_std[cols_mlr1]
X_test_mlr1 = X_test_std[cols_mlr1]

cols_mlr2 = ['type', 'density', 'quality',
             'residual sugar', 'total sulfur dioxide']
X_train_mlr2 = X_train_std[cols_mlr2]
X_test_mlr2 = X_test_std[cols_mlr2]

cols_mlr3 = ['type', 'density', 'quality', 'residual sugar',
             'type density', 'type residual sugar', 'type total sulfur dioxide']
X_train_mlr3 = X_train_std[cols_mlr3]
X_test_mlr3 = X_test_std[cols_mlr3]

cols_mlr4 = ['type', 'density', 'quality', 'residual sugar',
             'type density', 'type residual sugar', 'type total sulfur dioxide',
             'residual sugar^2', 'total sulfur dioxide^2', 'density^2']
X_train_mlr4 = X_train_std[cols_mlr4]
X_test_mlr4 = X_test_std[cols_mlr4]

# Check subsetted training set example
X_train_mlr4.head()

,type,density,quality,residual sugar,type density,type residual sugar,type total sulfur dioxide,residual sugar^2,total sulfur dioxide^2,density^2
0,-0.571351,1.672533,0.201750,2.666862,-0.571350,-0.486195,-0.443692,2.777606,0.618216,1.673675
1,1.750237,0.706277,-0.946655,-0.633169,1.750454,1.398768,1.681816,-0.465017,-0.990133,0.704860
2,-0.571351,0.906192,0.201750,1.881140,-0.571350,-0.486195,-0.443692,1.554719,-0.340899,0.905074
3,-0.571351,-1.692702,0.201750,-0.926506,-0.571350,-0.486195,-0.443692,-0.512705,-0.536270,-1.688304
4,-0.571351,-1.279545,0.201750,-0.088402,-0.571350,-0.486195,-0.443692,-0.272260,-0.981995,-1.277385


Now that we subsetted the training and test sets by the columns of interest, we are ready to fit and evaluate the models.

In [22]:
# Generate MLR models
cv_mlr1 = cross_validate(
    LinearRegression(),
    X_train_mlr1,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error",
    return_estimator = True)
print('MLR 1:', round(np.sqrt(-np.mean(cv_mlr1['test_score'])), 4))

cv_mlr2 = cross_validate(
    LinearRegression(),
    X_train_mlr2,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error",
    return_estimator = True)
print('MLR 2:', round(np.sqrt(-np.mean(cv_mlr2['test_score'])), 4))

cv_mlr3 = cross_validate(
    LinearRegression(),
    X_train_mlr3,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error",
    return_estimator = True)
print('MLR 3:', round(np.sqrt(-np.mean(cv_mlr3['test_score'])), 4))

cv_mlr4 = cross_validate(
    LinearRegression(),
    X_train_mlr4,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error",
    return_estimator = True)
print('MLR 4:', round(np.sqrt(-np.mean(cv_mlr4['test_score'])), 4))

MLR 1: 0.7639
MLR 2: 0.6965
MLR 3: 0.6751
MLR 4: 0.6374


Our most complicated model, MLR 4, did the best! That means we chose our interactions and polynomial features smartly.

Let's look at the intercepts and coefficients to see if they were stable over the various CV folds.

In [23]:
# Get intercept and coefficients for each fold
coefs = pd.DataFrame([est.coef_ for est in cv_mlr4['estimator']])
intercepts = pd.DataFrame([est.intercept_ for est in cv_mlr4['estimator']])

# Calculate mean values
mean_coef = np.mean(coefs, axis=0)
mean_intercept = np.mean(intercepts, axis=0)

# Print results
print("Mean Coefficients:\n")
for element in list(zip(X_train_mlr4.columns, mean_coef)):
    print(element)
print(coefs)
print("\nMean Intercept:\n", mean_intercept)
print(intercepts)

Mean Coefficients:

('type', -42.81727059210411)
('density', -111.02462005753299)
('quality', 0.1558475909563775)
('residual sugar', 1.0013884357298708)
('type density', 43.68356292461466)
('type residual sugar', 0.06739256559360171)
('type total sulfur dioxide', -0.1675639642725571)
('residual sugar^2', -0.5182197337227237)
('total sulfur dioxide^2', 0.012263457060306137)
('density^2', 109.64865192887218)
           0           1         2         3          4         5         6  \
0 -55.136524 -104.734096  0.155614  1.018341  56.029397  0.077463 -0.179867   
1 -41.496776 -102.246584  0.161913  0.976817  42.369529  0.032286 -0.165774   
2 -42.423763 -107.841539  0.167355  1.018665  43.272403  0.089749 -0.164555   
3 -48.578772 -103.493163  0.145492  1.012704  49.468364  0.067741 -0.165608   
4 -26.450518 -136.807719  0.148863  0.980415  27.278122  0.069724 -0.162017   

          7         8           9  
0 -0.515800  0.018456  103.320261  
1 -0.497630  0.001730  100.876346  
2 -0.55

The intercept was very similar for each case (about 10.5) but sometimes the coefficients varied significantly across the folds. The last fold (index=4) seemed to be the least similar.

Using the average intercept and coefficients, here's what our linear regression function would have looked like:

$\hat{y}$ = 10.5 - 42.8*`type` - 111.0*`density` + 0.156*`quality` + 1.00*`residual sugar` + 43.7*`type` * `density` + 0.0674*`type` * `residual sugar` - 0.168*`type` * `total sulfur dioxide` - 0.518*`residual sugar`$^2$ + 0.0123*`total sulfur dioxide`$^2$ + 109.6*`density`$^2$



#### LASSO Model

We will train a single LASSO model using a set of at least five predictors of our choice.

We will use CV to tune the LASSO parameter.

To give LASSO an equal footing with the best MLR model, we will provide all of the same input predictors.

However, if we had to choose only five parameters to include in the model, we would examine the best MLR model for the parameters with the greatest magnitudes. The interaction and polynomial terms would still be considered.

Top five parameters:
+ `density`
+ `density`$^2$
+ `type`*`density`
+ `type`
+ `residual sugar`

Even though these aren't five separate columns in the raw dataset, and even though there might be some multicollinearity, the previous work showed that there is value in including `density` along with its square and an interaction term associated with it. This is expected to be more beneficial than removing either `density` or its square and replacing it with an independent variable.  

We expect those five variables to have a significant contribution in the end, but perhaps the regularization will penalize those with inherent multicollinearity.

Now we will use CV to fit a LASSO model.

In [24]:
# Define input columns for LASSO
cols_LASSO = cols_mlr4
X_train_LASSO = X_train_std[cols_LASSO]

# Fit LASSO model
lasso_model = LassoCV(cv=5, random_state=42) \
                .fit(X_train_LASSO, y_train)

# Print fitted parameter and coefficients
print(lasso_model.alpha_)
print(lasso_model.intercept_)
pd.DataFrame(list(zip(X_train_LASSO.columns, lasso_model.coef_)))

0.0035329915929348627
10.498011032005584


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.952e+01, tolerance: 7.393e-01
  model = cd_fast.enet_coordinate_descent(


,0,1
0,type,0.000000
1,density,-1.265498
2,quality,0.205343
3,residual sugar,0.254130
4,type density,0.580628
5,type residual sugar,0.242228
6,type total sulfur dioxide,-0.138904
7,residual sugar^2,0.344583
8,total sulfur dioxide^2,-0.009137
9,density^2,-0.086194


The intercept was very similar to what we saw with MLR.

Unexpectedly, LASSO regularized the model by setting the coefficent for `type` to zero and shrinking the coefficent for `density`$^2$ way down. For MLR, `density`$^2$ was the second largest contributor (in terms of coefficient magnitude), second only to `density`. Perhaps due to the multiple collinearity with `density`, the model was able to determine that `density` could capture that effect and allow other independent variables to remain. The coefficient for `density` is still quite large in magnitude

Let's see how it scored relative to the MLR models.

In [25]:
# Print LASSO RMSE
mean_mse = np.mean(lasso_model.mse_path_, axis=1)
print("Mean MSE for best alpha:", round(np.min(mean_mse),4))
print("RMSE for best alpha:", round(np.sqrt(np.min(mean_mse)),4))

Mean MSE for best alpha: 0.4788
RMSE for best alpha: 0.6919


This doesn't look quite as good as our best MLR model. We'll confirm later when the best-of-class models are re-fit on the entire training sets and compared head-to-head.

#### Ridge Regression Model

We will train a single Ridge Regression model using a set of at least five predictors of our choice.

We will use CV to tune the Ridge Regression parameter.

To give Ridge Regression an equal footing with the best MLR model and the LASSO model, we will provide all of the same input predictors.

In [26]:
# Define input columns for Ridge Regression
cols_RR = cols_mlr4
X_train_RR = X_train_std[cols_RR]

# Fit Ridge Regression model
RR_model = RidgeCV(cv=5,
                   alphas=(0.0005, 0.0008, 0.001, 0.002, 0.003, 0.005,
                           0.01, 0.05, 0.1)) \
                .fit(X_train_RR, y_train)

# Print fitted parameter and coefficients
print(RR_model.alpha_)
print(RR_model.intercept_)
pd.DataFrame(list(zip(X_train_RR.columns, RR_model.coef_)))

0.002
10.498011032002383


,0,1
0,type,-41.970079
1,density,-88.101031
2,quality,0.163080
3,residual sugar,0.892513
4,type density,42.813086
5,type residual sugar,0.090378
6,type total sulfur dioxide,-0.166587
7,residual sugar^2,-0.373265
8,total sulfur dioxide^2,0.012366
9,density^2,86.703080


Once again, the intercept is very near 10.5.

The optimal value of alpha is 0.002.  

From the coefficients, we see that `density`, `density`$^2$, `type`, and `type`*`density` are the four most important parameters, just like in our MLR model.

This model class doesn't include mean error as a built-in metric, so we will print its best score (R^2) and leave the formal model metric comparison until later.

In [27]:
# Print Ridge Regression best score (R^2)
print("R^2 score for best alpha:", round(RR_model.best_score_,4))

R^2 score for best alpha: 0.7305


A value of $R^2$ = 0.7305 shows a moderate correlation between the model and actual data.

#### Elastic Net Model

We will train a single Elastic Net model using a set of at least five predictors of our choice.

We will use CV to tune the Elastic Net parameter.

To give Elastic Net an equal footing with the best MLR model, LASSO model, and Ridge Regression model, we will provide all of the same input predictors.

In [28]:
# Define input columns for Elastic Net
cols_EN = cols_mlr4
X_train_EN = X_train_std[cols_RR]

# Fit Elastic Net model
EN_model = ElasticNetCV(cv = 5,
                        random_state = 42,
                        max_iter=10000,
                        l1_ratio = [0.01, 0.02, 0.03, 0.04, 0.05,
                                    0.1, 0.25, 0.5, 0.75, 0.9,
                                    0.95, 0.97, 0.99, 1]) \
                .fit(X_train_EN, y_train)

# Print fitted parameter and coefficients
print(EN_model.alpha_)
print(EN_model.l1_ratio_)
print(EN_model.intercept_)
pd.DataFrame(list(zip(X_train_EN.columns, EN_model.coef_)))

0.04080834404491469
0.02
10.498011032005609


,0,1
0,type,0.228681
1,density,-0.618029
2,quality,0.221090
3,residual sugar,0.191851
4,type density,0.232316
5,type residual sugar,0.230616
6,type total sulfur dioxide,-0.108654
7,residual sugar^2,0.306327
8,total sulfur dioxide^2,-0.045152
9,density^2,-0.605131


The Elastic Net chose a L1 Ratio of 0.02, which means it weighs the Ridge Regression penalty much more than the LASSO penalty (which is unusual).

The intercept is very similar to all previous models (close to 10.5). The parameter coefficients are much less regularized and more evenly weighted here than with previous models, which is also unusual for a flexible-penalty method like this.

Let's look at how the RMSE compares with other models (early peek at our final model comparison).

In [29]:
# Print Elastic Net RMSE
mean_mse_EN = np.mean(EN_model.mse_path_, axis=1)
print("Mean MSE for best alpha:", round(np.min(mean_mse_EN),4))
print("RMSE for best parameters:", round(np.sqrt(np.min(mean_mse_EN)),4))

Mean MSE for best alpha: 0.5453
RMSE for best parameters: 0.7385


These metrics show that the hyperparameter optimization was unable to find something even as good as the MSR models at the beginning, or even the LASSO model. Perhaps it converged to a local minimum instead of a global minimum. We can test this by fitting a new Elastic Net model but only letting it explore high values of L1 Ratio.

In [30]:
# Trying again with high values of L1 Ratio

# Fit Elastic Net model
EN_model2 = ElasticNetCV(cv = 5,
                        random_state = 42,
                        max_iter=10000,
                        l1_ratio = [0.75, 0.9, 0.93, 0.94, 0.95,
                                    0.96, 0.97, 0.98, 0.99, 1]) \
                .fit(X_train_EN, y_train)

# Print fitted parameter and coefficients
print(EN_model2.alpha_)
print(EN_model2.l1_ratio_)
print(EN_model2.intercept_)
pd.DataFrame(list(zip(X_train_EN.columns, EN_model2.coef_)))

0.0034682945095592943
0.95
10.49801103200558


,0,1
0,type,0.000000
1,density,-1.352831
2,quality,0.205341
3,residual sugar,0.255494
4,type density,0.581739
5,type residual sugar,0.242553
6,type total sulfur dioxide,-0.139774
7,residual sugar^2,0.343948
8,total sulfur dioxide^2,-0.009223
9,density^2,-0.000000


In [31]:
# Print 2nd Elastic Net RMSE
mean_mse_EN2 = np.mean(EN_model2.mse_path_, axis=1)
print("Mean MSE for best alpha:", round(np.min(mean_mse_EN2),4))
print("RMSE for best parameters:", round(np.sqrt(np.min(mean_mse_EN2)),4))

Mean MSE for best alpha: 0.5453
RMSE for best parameters: 0.7385


Although the second Elastic Net model found a different optimum (much lower alpha, much higher L1 ratio, very different coefficients), the RMSE metric did not improve. This may indicate that it will perform the least well in a head-to-head comparison.

### Test Models

#### Re-Fit Best MLR Model

Before, the MLR models were only trained on a portion of the training set because one fold was withheld for cross-validation. The best-performing MLR model needs to be re-fit with the entire training set.

In [32]:
# Refit best MLR model
mlr_best = LinearRegression()
mlr_best.fit(X_train_mlr4, y_train)

# Check intercept and coefficients
print(mlr_best.intercept_)
print(pd.DataFrame(list(zip(X_train_mlr4.columns, mlr_best.coef_))))



10.498011032001752
                           0           1
0                       type  -45.593145
1                    density -105.216177
2                    quality    0.156332
3             residual sugar    1.007976
4               type density   46.466655
5        type residual sugar    0.065599
6  type total sulfur dioxide   -0.167720
7           residual sugar^2   -0.518172
8     total sulfur dioxide^2    0.013244
9                  density^2  103.826473


The intercept and coefficients relatively close to the mean values calculated from CV, so we should trust these values to be a good representation of the best MLR model fitted to the full training set.

#### Test using RMSE as the model metric

We will generate predictions using the test input dataset. We will save these values so we can use them again later.

Then, we will calculate the RMSE.

In [33]:
# Subset test inputs to match training inputs
X_test_subset = X_test_std[cols_mlr4]

# Predict using test set
y_mlr = mlr_best.predict(X_test_subset)
y_lasso = lasso_model.predict(X_test_subset)
y_RR = RR_model.predict(X_test_subset)
y_EN = EN_model.predict(X_test_subset)

# Determine RMSE for each model
RMSE_mlr = np.sqrt(mean_squared_error(y_mlr, y_test))
RMSE_lasso = np.sqrt(mean_squared_error(y_lasso, y_test))
RMSE_RR = np.sqrt(mean_squared_error(y_RR, y_test))
RMSE_EN = np.sqrt(mean_squared_error(y_EN, y_test))

# Print RMSEs
print('MLR RMSE:', round(RMSE_mlr, 5))
print('Lasso RMSE:', round(RMSE_lasso, 5))
print('RR RMSE:', round(RMSE_RR, 5))
print('EN RMSE:', round(RMSE_EN, 5))

MLR RMSE: 0.60363
Lasso RMSE: 0.66159
RR RMSE: 0.60681
EN RMSE: 0.66542


The MLR model happened to provide the best RMSE (lowest value), followed closely behind by Ridge Regression. It is interesting that the penalization methods did not perform as well, even though they had the same inputs. It is even more interesting that the Elastic Net model did not perform as well as the Ridge Regression model even though it could adapt to essentially become Ridge Regression by ignoring the L1 penalty term.

#### Test using MAE as the model metric

Now we will re-calcualte the metrics for MAE (without needing to re-predict the test set targets).

In [34]:
# Determine RMSE for each model
MAE_mlr = np.sqrt(mean_absolute_error(y_mlr, y_test))
MAE_lasso = np.sqrt(mean_absolute_error(y_lasso, y_test))
MAE_RR = np.sqrt(mean_absolute_error(y_RR, y_test))
MAE_EN = np.sqrt(mean_absolute_error(y_EN, y_test))

# Print RMSEs
print('MLR MAE:', round(MAE_mlr, 5))
print('Lasso MAE:', round(MAE_lasso, 5))
print('RR MAE:', round(MAE_RR, 5))
print('EN MAE:', round(MAE_EN, 5))

MLR MAE: 0.68845
Lasso MAE: 0.71819
RR MAE: 0.68969
EN MAE: 0.72399


Changing the metric still didn't change the best model selection. MLR still performed the best, followed very closely by Ridge Regression. Elastic Net was again the worst even though it theoretically could have been optimized to perform just as well as Lasso or Ridge Regression.

The MAE values exceed the corresponding RMSE for each model. This can happen because RMSE penalizes outliers more, but when the error is less than 1 (as can easily happen with standardized data), the square of the error can further shrink it before that operation is reversed on the mean sum of squared error by taking the square root. The MAE, on the other hand, linearly penalizes errors and is a more intuitive indication of how far off a prediction is from the true value, on average.

## Classification Task

The previous steps will be repeated while using a logistic regression model for classifying wine type.

But first, we need to figure out what variables are most important to focus on. This won't necessarily be the same as in the Regression part because the target variable has changed from `alcohol` to `type`. We will revisit the plots generated previously and analyze them for variables that can be used to separate white wines (`type` = 0) from red wines (`type` = 1).

We will generate new pairplots so we can see how they change now that we've standardized the data. This standardization cannot be used for training data because there would be data leakage, so we will standardize the training set again later after we engineer our features.

In [35]:
# Standardize full original dataset (excluding engineered features from Part I)
scaler2 = StandardScaler()
df_std = pd.DataFrame(scaler2.fit_transform(df.iloc[:,:12]),
                           columns=df.columns[:12])

# Add back in Type column (unstanderdized)
df_std['type'] = df['type']

df_std.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type
count,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6.497000e+03,6497.000000
mean,-3.849639e-16,1.049902e-16,2.187295e-17,3.499672e-17,1.749836e-17,-8.749179e-17,-6.999344e-17,-3.552167e-15,2.729744e-15,-5.424491e-16,9.974065e-16,-3.105959e-16,0.246114
std,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,1.000077e+00,0.430779
min,-2.634589e+00,-1.577330e+00,-2.192833e+00,-1.018034e+00,-1.342639e+00,-1.663583e+00,-1.941780e+00,-2.530192e+00,-3.100615e+00,-2.091935e+00,-2.089350e+00,-3.227687e+00,0.000000
25%,-6.289329e-01,-6.661613e-01,-4.723335e-01,-7.657978e-01,-5.147986e-01,-7.620742e-01,-6.855323e-01,-7.859527e-01,-6.748622e-01,-6.805919e-01,-8.316152e-01,-9.372296e-01,0.000000
50%,-1.660892e-01,-3.016939e-01,-5.941375e-02,-5.135612e-01,-2.578826e-01,-8.594301e-02,3.990667e-02,6.448888e-02,-5.287424e-02,-1.429373e-01,-1.608231e-01,2.079990e-01,0.000000
75%,3.738951e-01,3.664962e-01,4.911459e-01,5.584445e-01,2.559494e-01,5.901882e-01,7.122647e-01,7.648525e-01,6.313125e-01,4.619241e-01,6.776670e-01,2.079990e-01,0.000000
max,6.699425e+00,7.534354e+00,9.231281e+00,1.268682e+01,1.584219e+01,1.456357e+01,5.737257e+00,1.476879e+01,4.923029e+00,9.870879e+00,3.696231e+00,3.643685e+00,1.000000


In [36]:
''' # Commented out for Homework 8 to reduce runtime

# Create pairplot of standardized data
custom_palette = {1: 'red', 0: 'blue'}
sns.pairplot(df_std,
             hue='type',                # color points by type (white or red)
             plot_kws={'alpha': 0.3,    # increase transparency to see points stacking up
                       's': 5}          # decrease point size
             )

'''

" # Commented out for Homework 8 to reduce runtime\n\n# Create pairplot of standardized data\ncustom_palette = {1: 'red', 0: 'blue'}\nsns.pairplot(df_std,\n             hue='type',                # color points by type (white or red)\n             plot_kws={'alpha': 0.3,    # increase transparency to see points stacking up\n                       's': 5}          # decrease point size\n             )\n\n"

**Correlation Matrix:** The following variables have the largest magnitude of correlation with `type`:

- `total sulfur dioxide` (-0.7)
- `volatile acidity`(0.65)
- `chlorides` (0.51)

    These variables should be included in the analysis without interactions.

**Pairplot (Univariate):** The following univariate plots appear to show a significant partition between the two wine types (even if there is some overlap):
- `total sulfur dioxide`
- `volatile acidity`
- `chlorides`
- `residual sugar`
- `density`

    The first three of these align with the correlation matrix observations. The other two are less obvious because there is overlap between the regions, but nearly all of the red wine observations can be found to the left or right of a given threshold. These two variables would be great candidates for a decision tree boundary - because we are doing logistic regression instead, we will use these for polynomial terms as discussed below.

**Pairplot (Bivariate):** Many bivariate plots (scatter plots) appear to show a significant partition between the two wine types (even if there is some overlap). We will not consider cases where a vertical or horizontal line can separate the clusters because those represent univariate discrimination opportunities. We will use the following transformations to try to extract more information:

- **Radial features:** When one `type` is clusered near the origin and the other is clustered radially outward, we can use a radial feature based on the formula of a circle: $$x_1^2 + x_2^2$$
    - `fixed acidity` vs. `volatile acidity`
    - `fixed acidity` vs. `density`
    - `fixed acidity` vs. `sulphates`
    - `volatile acidity` vs. `sulphates`
    - `volatile acidity` vs. `density`
    - `sulphates` vs. `density`
    
    Because all four variables here seem to be radially separable from the others, they probably form a group that is radially separable in a higher dimension space. Because we standardized the data, we can probably use a simple circular radius formula instead of elliptical radius formula. Therefore, we will pack these four features into a **simple euclidian radius:** $$r^2 = x_1^2 + x_2^2 + x_3^2 + x_4^2$$

- **Ratio features:** When the two types appear to be separable by a line crossing through thte origin, a simple ratio of the variables should be helpful: $$x_1 / x_2$$
    [Strong]
    - `total sulfur dioxide` vs. `fixed acidity`
    - `total sulfur dioxide` vs. `volatile acidity`
    - `total sulfur dioxide` vs. `density`

    [Moderate or Weak]
    - `chlorides` vs. `free sulfur dioxide`
    - `free sulfur dioxide` vs. `volatile acidity`
    - `citric acid` vs. `fixed acidity`
    - `free sulfur dioxide` vs. `fixed acidity`

    Although the three most promising ratios involve `total sulfur dioxide` which will be accounted for on its own, we will keep those ratios because there is some overlap between classes in the univariate distribution and the diagonal decision boundary is expected to improve class discrimination.

- **Multiplication features:** When the two types of wine appear separable by a curved boundary, the variables can probably be multiplied to improve separation: $$x_1 * x_2$$
    - `pH` vs. `fixed acidity`

- **Polynomial features:**
    - `residual sugar`$^2$
        - We will square this term because the red wines have a tight grouping near zero while the white wines have a wider distribution with a big tail. The squaring should enhance separability.
    - `density`$^5$
        - We will calculate the fifth-order of this term because the red wines have a positive-leaning tight distribution and the white wines have a negative-leaning tight distribution. The cube will retain the original direction (right or left leaning) while further separating the distributions, but the densities are so narrowly grouped that we will get better discrimination by raising to the fifth power.

Because there are a lot of potential features to engineer, we need to be deliberate about which ones to include. We could have expanded the list as it was created, but we kept it somewhat limited to avoid unnecessary complexity. Here is a summary of the quantities of each type of feature we ensure is in our dataframe:

+ Univariate (untransformed): 3
+ Radial: 1
+ Ratio: 3
+ Multiplication: 1
+ Polynomial: 2
+ Total: 10 (7 new/engineered)

Because most of these are not standard "interaction" or "polynomial" features, we will add them column-by-column instead of using the `PolynomialFeatures` module. We will execute this below on the unstandardized data so we can restandardize later with only the training data.




In [37]:
# Add 10 engineered features to unstandardized dataframe

# Create copy of original dataframe
df2 = df.iloc[:,:13].copy()

# Add radial feature (simple euclidian distance)
df2['radial'] = np.sqrt((df2['fixed acidity'])**2 + (df2['volatile acidity'])**2 + (df2['sulphates'])**2 + (df2['density'])**2)

# Add ratio features
df2['ratio_tsd_fa'] = df2['total sulfur dioxide']/df2['fixed acidity']
df2['ratio_tsd_va'] = df2['total sulfur dioxide']/df2['volatile acidity']
df2['ratio_tsd_d'] = df2['total sulfur dioxide']/df2['density']

# Add multiplication features
df2['mult_pH_fa'] = df2['pH']*df2['fixed acidity']

# Add polynomial features
df2['poly_res_sugar'] = df2['residual sugar']**2
df2['poly_density'] = df2['density']**5

# Check statistics
df2.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,type,radial,ratio_tsd_fa,ratio_tsd_va,ratio_tsd_d,mult_pH_fa,poly_res_sugar,poly_density
count,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000,6497.000000
mean,7.215307,0.339666,0.318633,5.443235,0.056034,30.525319,115.744574,0.994697,3.218501,0.531268,10.491801,5.818378,0.246114,7.315876,16.827519,438.082003,116.357195,23.169805,52.262023,0.973852
std,1.296434,0.164636,0.145318,4.757804,0.035034,17.749400,56.521855,0.002999,0.160787,0.148806,1.192712,0.873255,0.430779,1.286715,8.667208,281.375543,56.733606,3.999886,96.455471,0.014727
min,3.800000,0.080000,0.000000,0.600000,0.009000,1.000000,6.000000,0.987110,2.720000,0.220000,8.000000,3.000000,0.000000,3.964179,0.491803,5.696203,6.018658,13.923000,0.360000,0.937190
25%,6.400000,0.230000,0.250000,1.800000,0.038000,17.000000,77.000000,0.992340,3.110000,0.430000,9.500000,5.000000,0.000000,6.506579,10.945946,202.040816,77.776992,20.604000,3.240000,0.962282
50%,7.000000,0.290000,0.310000,3.000000,0.047000,29.000000,118.000000,0.994890,3.210000,0.510000,10.300000,6.000000,0.000000,7.086998,17.464789,434.615385,118.980398,22.356000,9.000000,0.974710
75%,7.700000,0.400000,0.390000,8.100000,0.065000,41.000000,156.000000,0.996990,3.320000,0.600000,11.300000,6.000000,0.000000,7.784325,22.816901,624.000000,156.296964,24.696000,65.610000,0.985040
max,15.900000,1.580000,1.660000,65.800000,0.611000,289.000000,440.000000,1.038980,4.010000,2.000000,14.900000,9.000000,1.000000,15.957456,72.131148,1850.000000,443.039249,47.382000,4329.640000,1.210698


The basic statistics for new variables look believable and workable. None of the groupings are too tight, though the residual sugar variable has an enormous tail. That should make it easy to discriminate the red wine observations (near zero) from the white wine (anything with a large magnitude).

Now, let's do the training/test split and standardize the data based on the training set.

In [38]:
# Split training and test set
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    df2.drop('type', axis=1),
    df2['type'],
    test_size = 0.20,
    shuffle = True,
    random_state = 42,
    stratify = df2['type']
)

# Print observation counts and type proportions to check the split
print(len(X_train2))
print(len(X_test2))
print(y_train2.value_counts())


5197
1300
type
0    3918
1    1279
Name: count, dtype: int64


The split was successful.

In [39]:
# Standardize based on training set
scaler3 = StandardScaler()
X_train_std2 = pd.DataFrame(scaler3.fit_transform(X_train2),
                       columns=X_train2.columns)
X_test_std2 = pd.DataFrame(scaler3.transform(X_test2),
                       columns=X_test2.columns)

# Check stats to verify standardization
X_train_std2.describe()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,radial,ratio_tsd_fa,ratio_tsd_va,ratio_tsd_d,mult_pH_fa,poly_res_sugar,poly_density
count,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03,5.197000e+03
mean,6.836086e-18,8.750190e-17,-6.887356e-17,-2.802795e-17,-1.777382e-17,3.999110e-17,-1.025413e-16,-4.686752e-14,-2.156101e-15,2.050826e-17,-5.879034e-17,-1.517611e-16,-3.848716e-16,3.691486e-17,2.748106e-16,-1.497103e-16,5.509885e-16,3.007878e-17,-6.583834e-15
std,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00,1.000096e+00
min,-2.623567e+00,-1.563580e+00,-2.187852e+00,-1.010316e+00,-1.358711e+00,-1.664326e+00,-1.947703e+00,-2.522350e+00,-2.778159e+00,-2.118126e+00,-2.094424e+00,-3.243464e+00,-2.594725e+00,-1.889405e+00,-1.534116e+00,-1.950972e+00,-2.304577e+00,-5.191170e-01,-2.480096e+00
25%,-6.232249e-01,-6.602944e-01,-4.723414e-01,-7.588849e-01,-5.185759e-01,-7.603909e-01,-6.682412e-01,-7.997487e-01,-6.751579e-01,-6.902768e-01,-8.367689e-01,-9.466547e-01,-6.262994e-01,-6.677422e-01,-8.343878e-01,-6.634658e-01,-6.458403e-01,-4.902636e-01,-7.986608e-01
50%,-1.616075e-01,-2.989804e-01,-6.061888e-02,-5.074540e-01,-2.578442e-01,-8.243932e-02,4.257076e-02,3.989420e-02,-5.662815e-02,-1.463343e-01,-1.660197e-01,2.017497e-01,-1.730753e-01,7.546920e-02,-1.440879e-02,4.903429e-02,-2.013309e-01,-4.325569e-01,3.371902e-02
75%,3.769461e-01,3.634287e-01,5.569648e-01,5.611276e-01,2.636192e-01,5.955123e-01,7.000718e-01,7.729158e-01,6.237546e-01,4.656009e-01,6.724169e-01,2.017497e-01,3.665468e-01,6.846796e-01,6.556111e-01,7.054085e-01,3.722135e-01,1.345922e-01,7.673284e-01
max,6.685717e+00,7.469272e+00,9.203137e+00,1.265077e+01,1.608134e+01,1.460651e+01,5.764607e+00,1.476030e+01,4.891610e+00,9.848608e+00,3.690788e+00,3.646963e+00,6.704082e+00,6.403909e+00,5.002473e+00,5.786058e+00,6.046260e+00,4.285392e+01,1.606283e+01


The means are all near 0 and the standard deviations are all near 1.0, so it looks like it worked.

#### Train Models

##### Unregularized Logistic Regression Models

We are ready to define our first four un-regularized models.

+ LogReg Unregularized Model 1 (most simple)
    - Single Features: `total sulfur dioxide`, `volatile acidity`, `chlorides`
    - Interactions: None
    - Polynomials: None

+ LogReg Unregularized Model 2 (polynomial features without exponent, for comparison)
    - Single Features: `total sulfur dioxide`, `volatile acidity`, `chlorides`, `residual sugar`, `density`
    - Interactions: None
    - Polynomials: None

+ LogReg Unregularized Model 3 (polynomial features with exponent as planned)
    - Single Features: `total sulfur dioxide`, `volatile acidity`, `chlorides`
    - Interactions: None
    - Polynomials:
        - `residual sugar`$^2$
        - `density`$^5$

+ LogReg Unregularized Model 4 (most complex - includes all interactions and polynomials)
    - Single Features: `total sulfur dioxide`, `volatile acidity`, `chlorides`
    - Interactions (not strictly multiplicative):
        - `radial` (euclidian distance of four variables)
        - `pH` * `fixed acidity`
        - `total sulfur dioxide` / `fixed acidity`
        - `total sulfur dioxide` / `volatile acidity`
        - `total sulfur dioxide` / `density`

    - Polynomials:
        - `residual sugar`$^{0.5}$
        - `density`$^5$

Now we need to implement this in code.

In [40]:
# Subset training and test sets for each unregularized LogReg model
cols_lr1 = ['total sulfur dioxide', 'volatile acidity', 'chlorides']
X_train_lr1 = X_train_std2[cols_lr1]
X_test_lr1 = X_test_std2[cols_lr1]

cols_lr2 = ['total sulfur dioxide', 'volatile acidity', 'chlorides',
            'residual sugar', 'density']
X_train_lr2 = X_train_std2[cols_lr2]
X_test_lr2 = X_test_std2[cols_lr2]

cols_lr3 = ['total sulfur dioxide', 'volatile acidity', 'chlorides',
            'poly_res_sugar', 'poly_density']
X_train_lr3 = X_train_std2[cols_lr3]
X_test_lr3 = X_test_std2[cols_lr3]

cols_lr4 = ['total sulfur dioxide', 'volatile acidity', 'chlorides',
            'poly_res_sugar', 'poly_density',
            'radial',
            'mult_pH_fa',
            'ratio_tsd_fa', 'ratio_tsd_va', 'ratio_tsd_d']
X_train_lr4 = X_train_std2[cols_lr4]
X_test_lr4 = X_test_std2[cols_lr4]

# Check subsetted training set example
X_train_lr4.head()

,total sulfur dioxide,volatile acidity,chlorides,poly_res_sugar,poly_density,radial,mult_pH_fa,ratio_tsd_fa,ratio_tsd_va,ratio_tsd_d
0,0.753383,-0.780732,-0.402695,2.777606,1.676561,-0.481330,-0.376539,0.825001,1.112287,0.740551
1,-1.076958,1.447371,0.524351,-0.465017,0.700368,0.080079,0.524460,-1.074138,-1.218216,-1.080679
2,-0.117362,-1.262484,-0.141963,1.554719,0.901409,0.051568,0.050749,-0.217796,1.417374,-0.122756
3,-0.348376,-0.479637,-0.692397,-0.512705,-1.674661,-0.936810,-0.717970,-0.094102,-0.245675,-0.340078
4,-1.059188,-0.419418,-0.460635,-0.272260,-1.270525,0.289443,0.024792,-1.093335,-0.819210,-1.056935


Success!

Next step: fit and score the models.

In [41]:
# Generate Unregularized Logistic Regression models
cv_lr1 = cross_validate(
    LogisticRegression(penalty = None, solver = 'newton-cg'),
    X_train_lr1,
    y_train2,
    cv = 5,
    scoring = "neg_log_loss")
print('LR 1:', round(np.mean(cv_mlr1['test_score']), 4))

cv_lr2 = cross_validate(
    LogisticRegression(penalty = None, solver = 'newton-cg'),
    X_train_lr2,
    y_train2,
    cv = 5,
    scoring = "neg_log_loss")
print('LR 2:', round(np.mean(cv_mlr2['test_score']), 4))

cv_lr3 = cross_validate(
    LogisticRegression(penalty = None, solver = 'newton-cg'),
    X_train_lr3,
    y_train2,
    cv = 5,
    scoring = "neg_log_loss")
print('LR 3:', round(np.mean(cv_mlr3['test_score']), 4))

cv_lr4 = cross_validate(
    LogisticRegression(penalty = None, solver = 'newton-cg'),
    X_train_lr4,
    y_train2,
    cv = 5,
    scoring = "neg_log_loss",
    return_estimator = True)
print('LR 4:', round(np.mean(cv_lr4['test_score']), 4))
print()
print('LR 4 scores by fold:', cv_lr4['test_score'])

LR 1: -0.5836
LR 2: -0.4851
LR 3: -0.4558
LR 4: -0.0472

LR 4 scores by fold: [-0.04962192 -0.05057243 -0.04965335 -0.04659566 -0.03942958]


Wow! The added complexity of Model 4 really paid off! It has a much better score than the others (closest to zero by about an order of magnitude!). We also see somewhat consistent performance across the five folds - all are far better than the average score for other models.

Also, the difference between Model 2 and Model 3 shows that the polynomial representation of `density` and `residual sugar` actually did add value.

Lets look at the average intercepts and coefficients for the best model.

In [42]:
# Get intercept and coefficients for each fold
coefs = pd.DataFrame([est.coef_[0] for est in cv_lr4['estimator']])
intercepts = pd.DataFrame([est.intercept_ for est in cv_lr4['estimator']])

# Calculate mean values
mean_coef = np.mean(coefs, axis=0)
mean_intercept = np.mean(intercepts, axis=0)

# Print results
print("Mean Coefficients:\n")
for element in list(zip(X_train_lr4.columns, round(mean_coef, 5))):
    print(element)
print(round(coefs, 5))
print("\nMean Intercept:\n", round(mean_intercept, 5))
print(round(intercepts,5))

Mean Coefficients:

('total sulfur dioxide', -2.40428)
('volatile acidity', 0.75113)
('chlorides', 0.71871)
('poly_res_sugar', -4.38195)
('poly_density', 3.16825)
('radial', -2.09554)
('mult_pH_fa', 2.96855)
('ratio_tsd_fa', 2.88262)
('ratio_tsd_va', -1.63945)
('ratio_tsd_d', -2.33066)
         0        1        2        3        4        5        6        7  \
0 -2.13731  0.65260  0.66186 -4.81516  3.15441 -2.22730  3.13777  2.56173   
1 -2.38469  0.81769  0.79993 -4.48144  3.40340 -2.09826  2.86160  2.54517   
2 -2.19667  0.82529  0.62381 -4.39722  3.17490 -1.93600  2.53722  2.39302   
3 -2.97487  0.65519  0.77740 -4.28241  3.09189 -2.20273  3.40843  4.28248   
4 -2.32787  0.80490  0.73057 -3.93352  3.01665 -2.01340  2.89775  2.63070   

         8        9  
0 -1.93537 -2.07644  
1 -1.19706 -2.30877  
2 -1.67741 -2.12351  
3 -1.78004 -2.90374  
4 -1.60735 -2.24086  

Mean Intercept:
 0   -4.69459
dtype: float64
         0
0 -4.84566
1 -4.59858
2 -4.83745
3 -4.59528
4 -4.59598


While there was some fold-to-fold variability in intercepts and coefficients, they didn't not vary wildly.

When looking at the mean coefficients, we see that the most important (most heavily weighted) terms were for `pH`*`fixed acidity`, `total sulfur dioxide`/`fixed acidity`, and the `radial` parameter (euclidian distance between four variables).

None of the ten coefficients are very small, meaning that we overall chose good features to go into the model! This is reinforced by the fact that _almost_ all of the engineered features were weighted more heavily than the independent, original variables.

##### L1-Regularized Logistic Regression Model

We will apply the concept of the LASSO penalty method to the Logistic Regression model by specifying the L1 penalty.

To give the L1-regularized model an equal footing with the best unregularized Logistic Regression model, we will provide all of the same input predictors.

Now we will use CV to fit an L1-regularized model.

In [43]:
# Define input columns for L1-regularized model
cols_L1 = cols_lr4
X_train_L1 = X_train_std2[cols_L1]
X_test_L1 = X_test_std2[cols_L1]

# Fit L1-regularized model
L1_model = LogisticRegressionCV(cv=5,
                                solver = 'saga',
                                penalty = 'l1',
                                Cs = 100,
                                max_iter = 1000,
                                scoring = 'neg_log_loss',
                                random_state=42)
L1_model.fit(X_train_L1, y_train2)

LogisticRegressionCV(Cs=100, cv=5, max_iter=1000, penalty='l1', random_state=42,
                     scoring='neg_log_loss', solver='saga')

In [44]:
# Print fitted parameter and coefficients
print('Best C =', round(L1_model.C_[0], 5))
print('intercept =', round(L1_model.intercept_[0], 5))
pd.DataFrame(list(zip(X_train_L1.columns, L1_model.coef_[0])))

Best C = 12.32847
intercept = -4.69166


,0,1
0,total sulfur dioxide,-1.073246
1,volatile acidity,0.698487
2,chlorides,0.706324
3,poly_res_sugar,-4.321351
4,poly_density,3.124274
5,radial,-2.231011
6,mult_pH_fa,2.891496
7,ratio_tsd_fa,2.092630
8,ratio_tsd_va,-1.701579
9,ratio_tsd_d,-2.868832


The coefficients and intercept are very similar to the best unregularized model. Let's see how the score compares (as an early look for the final comparison we will eventually do on the test set).

In [45]:
#Find mean score for best value of C
scores = L1_model.scores_[1]
mean_scores = scores.mean(axis=0)
best_C_index = mean_scores.argmax()
best_C = L1_model.Cs_[best_C_index]
best_score = mean_scores[best_C_index]

print('Best C:', round(best_C, 5))
print('Mean CV score for best C', round(best_score, 5))

Best C: 12.32847
Mean CV score for best C -0.04715


This is better than the best unregularized model, but just by a hair. Looks like regularization helped (marginally).

##### L2-Regularized Logistic Regression Model

We will apply the concept of the Ridge Regression penalty method to the Logistic Regression model by specifying the L2 penalty.

To give the L2-regularized model an equal footing with the best unregularized Logistic Regression model, we will provide all of the same input predictors.

Now we will use CV to fit an L2-regularized model.

In [46]:
# Define input columns for L2-regularized model
cols_L2 = cols_lr4
X_train_L2 = X_train_std2[cols_L2]
X_test_L2 = X_test_std2[cols_L2]

# Fit L2-regularized model
L2_model = LogisticRegressionCV(cv=5,
                                solver = 'saga',
                                penalty = 'l2',
                                Cs = 100,
                                max_iter = 1000,
                                scoring = 'neg_log_loss',
                                random_state=42)
L2_model.fit(X_train_L2, y_train2)

LogisticRegressionCV(Cs=100, cv=5, max_iter=1000, random_state=42,
                     scoring='neg_log_loss', solver='saga')

In [47]:
# Print fitted parameter and coefficients
print('Best C =', round(L2_model.C_[0], 5))
print('intercept =', round(L2_model.intercept_[0], 5))
pd.DataFrame(list(zip(X_train_L2.columns, L2_model.coef_[0])))

Best C = 7.0548
intercept = -4.56279


,0,1
0,total sulfur dioxide,-1.895407
1,volatile acidity,0.725878
2,chlorides,0.705596
3,poly_res_sugar,-4.067019
4,poly_density,3.055986
5,radial,-2.163264
6,mult_pH_fa,2.775638
7,ratio_tsd_fa,1.852005
8,ratio_tsd_va,-1.608122
9,ratio_tsd_d,-1.867810


The intercept and coefficients again aren't too far off from the unregularized values. Let's check out the average scores for the best L2-regularized model.

In [48]:
# Find mean score for best value of C
scores2 = L2_model.scores_[1]
mean_scores2 = scores2.mean(axis=0)
best_C_index2 = mean_scores2.argmax()
best_C2 = L2_model.Cs_[best_C_index2]
best_score2 = mean_scores2[best_C_index2]

print('Best C:', round(best_C2, 5))
print('Mean CV score for best C', round(best_score2, 5))

Best C: 7.0548
Mean CV score for best C -0.047


The mean score is just a hair better again, but this could be in the noise! We need to wait until the final evaluation to make conclusions.

##### Elastic Net-Regularized Logistic Regression Model

We will apply the concept of the Elastic Net penalty method to the Logistic Regression model by specifying the `elasticnet` penalty.

To give the Elastic Net-regularized model an equal footing with the best unregularized Logistic Regression model, we will provide all of the same input predictors.

Now we will use CV to fit an Elastic Net-regularized model.

In [49]:
# Define input columns for Elastic Net-regularized model
cols_EN = cols_lr4
X_train_EN = X_train_std2[cols_EN]
X_test_EN = X_test_std2[cols_EN]

# Fit L2-regularized model
EN_LogReg_model = LogisticRegressionCV(cv=5,
                                solver = 'saga',
                                penalty = 'elasticnet',
                                Cs = 25,
                                l1_ratios = [0.02, 0.05, 0.1,
                                            0.9, 0.95, 0.98],
                                max_iter = 2000,
                                scoring = 'neg_log_loss',
                                random_state=42)
EN_LogReg_model.fit(X_train_EN, y_train2)

LogisticRegressionCV(Cs=25, cv=5, l1_ratios=[0.02, 0.05, 0.1, 0.9, 0.95, 0.98],
                     max_iter=2000, penalty='elasticnet', random_state=42,
                     scoring='neg_log_loss', solver='saga')

In [50]:
# Print fitted parameter and coefficients
print('Best C =', round(EN_LogReg_model.C_[0], 5))
print('Best L1 Ratio =', EN_LogReg_model.l1_ratio_[0])
print('intercept =', round(EN_LogReg_model.intercept_[0], 5))
pd.DataFrame(list(zip(X_train_EN.columns, EN_LogReg_model.coef_[0])))

Best C = 4.64159
Best L1 Ratio = 0.02
intercept = -4.52658


,0,1
0,total sulfur dioxide,-1.817630
1,volatile acidity,0.727598
2,chlorides,0.704427
3,poly_res_sugar,-3.983940
4,poly_density,3.026033
5,radial,-2.148678
6,mult_pH_fa,2.722988
7,ratio_tsd_fa,1.684755
8,ratio_tsd_va,-1.591703
9,ratio_tsd_d,-1.794831


The intercept and coefficients are, once again, close to what we saw previously. This is not surprising because the Elastic Net penalty is on a sliding scale between L1 and L2 regularization. Let's check out the average scores.

In [51]:
# Fit model again, only using optimized L1 Ratio and C values
#    This will help simplify the search for the best-case scores

# Re-Fit L2-regularized model
EN_LogReg_model = LogisticRegressionCV(cv=5,
                                solver = 'saga',
                                penalty = 'elasticnet',
                                Cs = [2.15443],
                                l1_ratios = [0.02],
                                max_iter = 1000,
                                scoring = 'neg_log_loss',
                                random_state=42)
EN_LogReg_model.fit(X_train_EN, y_train2)

EN_LogReg_model.scores_

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


{np.int64(1): array([[[-0.04929553]],
 
        [[-0.05177721]],
 
        [[-0.05159174]],
 
        [[-0.04218219]],
 
        [[-0.04147745]]])}

In [52]:
np.mean(EN_LogReg_model.scores_[1])

np.float64(-0.04726482202478842)

The average score is virtually identical to the L2-regularized model, which makes sense because the L1 Ratio was optimized to 0.02 (the lowest value in the list provided), which basically makes it L2-regularized.

### Test Models

#### Re-Fit Best Unregularized Logistic Regression Model

Before, the unregularized Logistic Regression models were only trained on a portion of the training set because one fold was withheld for cross-validation. The best-performing unregularized Logistic Regression model needs to be re-fit with the entire training set.

In [53]:
# Refit best unregularized Logistic Regression model
LogReg_unreg_best = LogisticRegression(penalty = None, solver = 'newton-cg')
LogReg_unreg_best.fit(X_train_lr4, y_train2)

# Check intercept and coefficients
print(LogReg_unreg_best.intercept_)
print(pd.DataFrame(list(zip(X_train_lr4.columns, LogReg_unreg_best.coef_[0]))))

[-4.67503184]
                      0         1
0  total sulfur dioxide -2.386967
1      volatile acidity  0.752232
2             chlorides  0.710496
3        poly_res_sugar -4.385409
4          poly_density  3.171897
5                radial -2.068056
6            mult_pH_fa  2.917770
7          ratio_tsd_fa  2.820196
8          ratio_tsd_va -1.604458
9           ratio_tsd_d -2.303832


These values are similar to the CV-fold-averaged values we saw earlier, so it looks like we successfully re-trained the best model on the full training set.

#### Test using Negative Log Loss as the model metric

We will generate predictions using the test input dataset. We will save these values so we can use them again later.

Then, we will calculate the Log Loss.

In [54]:
# Subset test inputs to match training inputs
X_test_subset2 = X_test_std2[cols_lr4]

# Predict using test set
y_LogReg_unreg = LogReg_unreg_best.predict_proba(X_test_subset2)
y_LogReg_L1 = L1_model.predict_proba(X_test_subset2)
y_LogReg_L2 = L2_model.predict_proba(X_test_subset2)
y_LogReg_EN = EN_LogReg_model.predict_proba(X_test_subset2)

In [55]:
# Determine Log Loss for each model
LL_LogReg_unreg = log_loss(y_test2, y_LogReg_unreg[:,1])
LL_LogReg_L1 = log_loss(y_test2, y_LogReg_L1[:,1])
LL_LogReg_L2 = log_loss(y_test2, y_LogReg_L2[:,1])
LL_LogReg_EN = log_loss(y_test2, y_LogReg_EN[:,1])

# Print Log Losses
print('Unreg Log Loss:', round(LL_LogReg_unreg, 5))
print('L1 Log Loss:', round(LL_LogReg_L1, 5))
print('L2 Log Loss:', round(LL_LogReg_L2, 5))
print('EN Log Loss:', round(LL_LogReg_EN, 5))

Unreg Log Loss: 0.04058
L1 Log Loss: 0.04033
L2 Log Loss: 0.04026
EN Log Loss: 0.04038


The L2-penalized Logarithmic Regression model did the best based on the Log Loss score! However, it is a very close call; all models did very well.

_Note that I have an un-specified random state somewhere. My results change when I run the model again. However, the results only change a very small amount. All models here are virtually tied._

#### Test using Accuracy as the model metric

Now we will calculate the Accuracy metric for comparison (without needing to re-predict the test set targets).

Unfortunately, we can't use our `predict_proba` results (which are probabilities expressed as a decimal) when comparing the accuracy of binary values. First, we need to convert the probabilities back to predictions of 0 or 1.

In [56]:
# Convert probabilities to binary result
y_LogReg_unreg_binary = (y_LogReg_unreg[:,1] >= 0.5).astype(int)
y_LogReg_L1_binary = (y_LogReg_L1[:,1] >= 0.5).astype(int)
y_LogReg_L2_binary = (y_LogReg_L2[:,1] >= 0.5).astype(int)
y_LogReg_EN_binary = (y_LogReg_EN[:,1] >= 0.5).astype(int)

y_LogReg_unreg_binary

array([0, 0, 0, ..., 0, 0, 0])

In [57]:

# Determine Accuracy for each model
Acc_LogReg_unreg = accuracy_score(y_test2, y_LogReg_unreg_binary)
Acc_LogReg_L1 = accuracy_score(y_test2, y_LogReg_L1_binary)
Acc_LogReg_L2 = accuracy_score(y_test2, y_LogReg_L2_binary)
Acc_LogReg_EN = accuracy_score(y_test2, y_LogReg_EN_binary)

# Print Accuracy
print('Unreg Accuracy:', round(Acc_LogReg_unreg, 5))
print('L1 Accuracy:', round(Acc_LogReg_L1, 5))
print('L2 Accuracy:', round(Acc_LogReg_L2, 5))
print('EN Accuracy:', round(Acc_LogReg_EN, 5))

Unreg Accuracy: 0.99154
L1 Accuracy: 0.99154
L2 Accuracy: 0.99077
EN Accuracy: 0.99077


The Accuracy score winner is slightly different than the winner from the Log Loss scores. The unregularized and L1-penalized models have the best score, while the others are very slightly lower.

The difference is miniscule. All of them did extremely well, much better than expected. We should be very satisfied with the results!

# New Homework 8 Content

Below, all of the content is new for Homework 8.

The purpose of this homework is to practice fitting tree-based models using `scikit-learn`.

The dataset and response variables for regression and classification are the same as in Homework 7.

## Regression Task
(`alcohol` as Response)

### Train Models

In [58]:
# Import relevant modules
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, \
                            plot_tree
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import GridSearchCV

#### Regression Tree

This tree will use at least five predictors and use CV to select tuning parameters (`max_depth` and `min_samples_leaf`)

To have a more fair comparision between the tree-based models and the linear and logistic models used in Homework 7, we will supply the same predictors.

In [59]:
# Select predictors for decision tree
cols_reg_tree = cols_mlr4
X_train_reg_tree = X_train_std[cols_reg_tree]
X_test_reg_tree = X_test_std[cols_reg_tree]
cols_reg_tree

['type',
 'density',
 'quality',
 'residual sugar',
 'type density',
 'type residual sugar',
 'type total sulfur dioxide',
 'residual sugar^2',
 'total sulfur dioxide^2',
 'density^2']

We will select a wide array of potential `max_depth` and `min_samples_leaf` values to explore via grid search CV. This shouldn't take long to run because of the efficiency of the decision tree fitting algorithm.  

In [60]:
# Define parameter space for grid search
parameters = {'max_depth': [2, 3, 4, 5, 6, 7, 8],
              'min_samples_leaf': [2, 5, 10, 15, 25, 50, 75, 100]
              }

# Fit tree model with grid search CV
grid_reg_tree = GridSearchCV(
    DecisionTreeRegressor(random_state = 42),
    parameters,
    cv = 5,
    scoring = 'neg_mean_squared_error',
    refit = True)
grid_reg_tree.fit(X_train_reg_tree, y_train)

# Print best model hyperparameters
print(grid_reg_tree.best_estimator_)
print(grid_reg_tree.best_score_, grid_reg_tree.best_params_)

# Print RMSE
print('RMSE: ', np.sqrt(-grid_reg_tree.best_score_))

# Note: because refit = True, the model was automatically refit on the full training set

DecisionTreeRegressor(max_depth=8, min_samples_leaf=5, random_state=42)
-0.35157647378697365 {'max_depth': 8, 'min_samples_leaf': 5}
RMSE:  0.5929388448963128


The RMSE is quite impressive compared to the previous models explored in Homework 7.

The large depth (8) and small minimum samples per leaf (5) has me worried that the model might be overfitting, but because we used cross-validation, it is probably fine. We will see if the end results with the test set are drastically different than with the training set.

#### Random Forest (Regression)

We will repeat the previous step using the same predictors but only cross validate to determine `max_features`. The other hyperparameters can be freely explored by the unconstrained fitting algorithm.

In [61]:
# Define parameter space for grid search
parameters = {'max_features': [2, 3, 4, 5, 6, 7, 8, 9, 10]}

# Fit random forest with grid search CV
grid_reg_forest = GridSearchCV(
    RandomForestRegressor(random_state = 42, n_estimators = 500),
    parameters,
    cv = 5,
    scoring = 'neg_mean_squared_error',
    refit = True)
grid_reg_forest.fit(X_train_reg_tree, y_train) # intentionally using same training set

# Print best model hyperparameters
print(grid_reg_forest.best_estimator_)
print(grid_reg_forest.best_score_, grid_reg_forest.best_params_)

# Print RMSE
print('RMSE: ', np.sqrt(-grid_reg_forest.best_score_))

# Note: because refit = True, the model was automatically refit on the full training set

RandomForestRegressor(max_features=2, n_estimators=500, random_state=42)
-0.2566331247895306 {'max_features': 2}
RMSE:  0.5065897006350707


As expected, the random forest did better than the single tree. However, I did not expect that the best model would have a `max_features` value of 2. I expected that 3-5 features per tree would be optimal!

### Test Models

We will use these two models to compare performance against each other and against the four regression models from Homework 7 (transcribing the prior results below). These comparisions will use RMSE and MAE as the model metric.

#### RMSE as the model metric

In [62]:
# Subset test inputs to match training inputs
X_test_subset = X_test_std[cols_reg_tree]

# Predict using test set
y_reg_tree = grid_reg_tree.predict(X_test_subset)
y_reg_forest = grid_reg_forest.predict(X_test_subset)

# Determine RMSE for each model
RMSE_reg_tree = np.sqrt(mean_squared_error(y_reg_tree, y_test))
RMSE_reg_forest = np.sqrt(mean_squared_error(y_reg_forest, y_test))

# Print RMSEs
print('Tree RMSE:', round(RMSE_reg_tree, 5))
print('Forest RMSE:', round(RMSE_reg_forest, 5))

Tree RMSE: 0.58209
Forest RMSE: 0.46644


Here are the two new test scores and the four previous test scores, all consolidated in one place:

_Tree-Based Models:_
+ Tree RMSE: .......0.58209
+ Forest RMSE: .. 0.46644

_Linear Regression Models:_
+ MLR RMSE: ...... 0.60363
+ Lasso RMSE: ...  0.66159
+ RR RMSE: ......... 0.60681
+ EN RMSE: ......... 0.66542

The tree-based models performed far better than the other models, and the random forest performed far better than the single tree regression model. This demonstrates the value of this model class.

#### MAE as the model metric

In [63]:
# Don't need to predict again, just calculate new metric

# Determine MAE for each model
MAE_reg_tree = np.sqrt(mean_absolute_error(y_reg_tree, y_test))
MAE_reg_forest = np.sqrt(mean_absolute_error(y_reg_forest, y_test))

# Print RMSEs
print('Tree MAE:', round(MAE_reg_tree, 5))
print('Forest MAE:', round(MAE_reg_forest, 5))

Tree MAE: 0.6649
Forest MAE: 0.57954


Here are the two new test scores and the four previous test scores, all consolidated in one place:

_Tree-Based Models:_
+ Tree MAE: .......0.66490
+ Forest MAE: ... 0.57954

_Linear Regression Models:_
+ MLR MAE: ...... 0.68845
+ Lasso MAE: ...  0.71819
+ RR MAE: ......... 0.68969
+ EN MAE: ......... 0.72399

Again, the tree-based models performed far better than the other models, and the random forest performed far better than the single tree regression model. However, the magnitude of improvement is not as pronounced, suggesting that the tree-based models are tuned to better handle outliers that would have contributed more significantly to RMSE than to MAE.  

## Classification Task
(Wine `type` as Response)

This section will repeat the previous section but with classification trees and forests of classification trees.

### Train Models

#### Classification Tree

This tree will use at least five predictors and use CV to select tuning parameters (`max_depth` and `min_samples_leaf`)

To have a more fair comparision between the tree-based models and the linear and logistic models used in Homework 7, we will supply the same predictors.

In [64]:
# Select predictors for decision tree
cols_cls_tree = cols_lr4
X_train_cls_tree = X_train_std2[cols_cls_tree]
X_test_cls_tree = X_test_std2[cols_cls_tree]
cols_cls_tree

['total sulfur dioxide',
 'volatile acidity',
 'chlorides',
 'poly_res_sugar',
 'poly_density',
 'radial',
 'mult_pH_fa',
 'ratio_tsd_fa',
 'ratio_tsd_va',
 'ratio_tsd_d']

We will select a wide array of potential `max_depth` and `min_samples_leaf` values to explore via grid search CV. These will match the values explored for the regression tree. This shouldn't take long to run because of the efficiency of the decision tree fitting algorithm.

In [65]:
# Define parameter space for grid search
parameters = {'max_depth': [2, 3, 4, 5, 6, 7, 8],
              'min_samples_leaf': [2, 5, 10, 15, 25, 50, 75, 100]
              }

# Fit tree model with grid search CV
grid_cls_tree = GridSearchCV(
    DecisionTreeClassifier(random_state = 42),
    parameters,
    cv = 5,
    scoring = 'neg_log_loss',
    refit = True)
grid_cls_tree.fit(X_train_cls_tree, y_train2)

# Print best model hyperparameters
print(grid_cls_tree.best_estimator_)
print(grid_cls_tree.best_score_, grid_cls_tree.best_params_)

# Print Log Loss
print('Negative Log Loss: ', grid_cls_tree.best_score_)

# Note: because refit = True, the model was automatically refit on the full training set

DecisionTreeClassifier(max_depth=3, min_samples_leaf=25, random_state=42)
-0.06841582392251334 {'max_depth': 3, 'min_samples_leaf': 25}
Negative Log Loss:  -0.06841582392251334


The Negative Log Loss metric is quite impressive compared to the first three Logistic Regression classifiers we attempted in Homework 7, but it doesn't look as good as the best (in the initial set of four unregularized models) or the subsequent regularized models.

The shallow depth (3) and moderate  minimum samples per leaf (25) gives me confidence that the model is not over-fit to the training data. This tree has a very different size than the regression tree did, perhaps because it only has two possible responses.

#### Random Forest (Classification)

We will repeat the previous step using the same predictors but only cross validate to determine `max_features`. The other hyperparameters can be freely explored by the unconstrained fitting algorithm.

In [66]:
# Define parameter space for grid search
parameters = {'max_features': [2, 3, 4, 5, 6, 7, 8, 9, 10]}

# Fit random forest with grid search CV
grid_cls_forest = GridSearchCV(
    RandomForestClassifier(random_state = 42, n_estimators = 500),
    parameters,
    cv = 5,
    scoring = 'neg_log_loss',
    refit = True)
grid_cls_forest.fit(X_train_cls_tree, y_train2) # intentionally using same training set

# Print best model hyperparameters
print(grid_cls_forest.best_estimator_)
print(grid_cls_forest.best_score_, grid_cls_forest.best_params_)

# Print Negative Log Loss
print('Negative Log Loss: ', grid_cls_forest.best_score_)

# Note: because refit = True, the model was automatically refit on the full training set

RandomForestClassifier(max_features=4, n_estimators=500, random_state=42)
-0.04646855334754314 {'max_features': 4}
Negative Log Loss:  -0.04646855334754314


As expected, the random forest did better than the single tree. Also, the `max_features` value of 4 seems like it makes more sense here than the value of 2 for Regression.

### Test Models

We will use these two models to compare performance against each other and against the four classification models from Homework 7 (transcribing the prior results below). These comparisions will use Negative Log Loss and Accuracy as the model metrics.

#### Negative Log Loss as the model metric

In [67]:
# Subset test inputs to match training inputs
X_test_subset2 = X_test_std2[cols_cls_tree]

# Predict using test set
y_cls_tree = grid_cls_tree.predict_proba(X_test_subset2)
y_cls_forest = grid_cls_forest.predict_proba(X_test_subset2)

# Determine Negative Log Loss for each model
NegLogLoss_cls_tree = -log_loss(y_test2, y_cls_tree[:,1])
NegLogLoss_cls_forest = -log_loss(y_test2, y_cls_forest[:,1])

# Print Negative Log Losses
print('Tree NegLogLoss:', round(NegLogLoss_cls_tree, 5))
print('Forest NegLogLoss:', round(NegLogLoss_cls_forest, 5))

Tree NegLogLoss: -0.04326
Forest NegLogLoss: -0.01779


Here are the two new classification test scores and the four previous test scores, all consolidated in one place:

_Tree-Based Models:_
+ Tree Neg Log Loss: ......... -0.04326
+ Forest Neg Log Loss: ..... -0.01779

_Logistic Regression Models:_
+ LogReg Neg Log Loss: .. -0.04058
+ L1 Neg Log Loss: ............  -0.04033
+ L2 Neg Log Loss: ........... -0.04026
+ EN Neg Log Loss: ........... -0.04038


The single classification tree model performed nearly as well as the logistic regression models. However, the random forest with classification trees performed better than any of them!

#### Accuracy as the model metric

Now we will calculate the Accuracy metric for comparison (without needing to re-predict the test set targets).

Unfortunately, we can't use our `predict_proba` results (which are probabilities expressed as a decimal) when comparing the accuracy of binary values. First, we need to convert the probabilities back to predictions of 0 or 1.

In [68]:
# Convert probabilities to binary result
y_cls_tree_binary = (y_cls_tree[:,1] >= 0.5).astype(int)
y_cls_forest_binary = (y_cls_forest[:,1] >= 0.5).astype(int)

# Check output format
y_cls_tree_binary

array([0, 0, 0, ..., 0, 0, 0])

In [69]:
# Determine Accuracy for each model
Acc_cls_tree = accuracy_score(y_test2, y_cls_tree_binary)
Acc_cls_forest = accuracy_score(y_test2, y_cls_forest_binary)


# Print Accuracy
print('Cls Tree Accuracy:', round(Acc_cls_tree, 5))
print('Cls Forest Accuracy:', round(Acc_cls_forest, 5))

Cls Tree Accuracy: 0.98692
Cls Forest Accuracy: 0.99385


Here are the two new classification accuracy test scores and the four previous test scores, all consolidated in one place:

_Tree-Based Models:_
+ Tree Accuracy: ......... 0.98692
+ Forest Accuracy: ...... 0.99385

_Logistic Regression Models:_
+ LogReg Accuracy: .. 0.99154
+ L1 Accuracy: ............ 0.99154
+ L2 Accuracy: ........... 0.99077
+ EN Accuracy: ........... 0.99077

Again, the single classification tree model performed nearly as well as the logistic regression models. And yet again, the random forest with classification trees performed the best.